In [ ]:
#Guardar los datos
#La IDE permite observarlos, pero conviene usar un programa que guarde directamente el puerto serie en un archivo CSV.

#pip install pyserial

import serial
import csv
import time

PORT = "COM5"       # Cambiar segun la PC
BAUD = 115200
DURATION_S = 60

OUTPUT = "ruido_hcsr04_centro.csv"

with serial.Serial(
    PORT,
    BAUD,
    timeout=2
) as ser:

    # Arduino Uno suele reiniciarse al abrir el puerto
    time.sleep(2)

    # Descartar datos iniciales
    ser.reset_input_buffer()

    with open(
        OUTPUT,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "t_us",
            "distancia_cm",
            "valid",
            "echo_us"
        ])

        start = time.monotonic()
        count = 0

        while time.monotonic() - start < DURATION_S:

            line = ser.readline().decode(
                errors="ignore"
            ).strip()

            parts = line.split(",")

            if len(parts) != 4:
                continue

            if parts[0] == "t_us":
                continue

            try:
                t_us = int(parts[0])
                distancia = float(parts[1])
                valid = int(parts[2])
                echo_us = int(parts[3])

            except ValueError:
                continue

            writer.writerow([
                t_us,
                distancia,
                valid,
                echo_us
            ])

            count += 1

print(f"Archivo guardado: {OUTPUT}")
print(f"Muestras registradas: {count}")
